# Downloading from public repositories

Three places where pathogen-genomic data lives in the open, and how
to pull from each:

- **NCBI SRA**, the canonical archive for raw sequencing reads
- **ENA** (European Nucleotide Archive), a mirror of SRA with simpler
  HTTPS-based downloads
- **GenBank**, for reference genomes and annotated sequences

Takes about five minutes with the default settings. The default
accession is a small E. coli dataset (~50 MB) that downloads
quickly.

> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (Local)`** (under "Start python Kernel"). Don't pick `Python 3 (ipykernel) (Local)`, PyTorch, or TensorFlow — those are missing libraries this notebook needs and will fail at the first GCS read with `ModuleNotFoundError`. If you already see `Python 3 (Local)` in the top-right of this tab, you're good to go.

In [ ]:
import subprocess

DETECTED_PROJECT = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# A deliberately small accession so the notebook completes quickly.
# Override with whatever SRA accession you actually want.
ACCESSION  = "SRR2584863"          # @param {type:"string"}  -- ~50 MB E. coli WGS
OUTPUT_DIR = "./public-downloads"  # @param {type:"string"}
PROJECT_ID = DETECTED_PROJECT      # @param {type:"string"}

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Downloads will land in: {OUTPUT_DIR}")

In [ ]:
import subprocess

active_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()

print(f"GCP project: {PROJECT_ID}")
print(f"Identity:    {active_account}")

## Setup: install sra-tools

`sra-tools` is NCBI's official toolkit for downloading from SRA. We
use two of its commands:

- `prefetch <accession>` pulls the SRA archive file
- `fasterq-dump <accession>` converts that archive into a normal
  FASTQ

The cell checks first and skips the install if `sra-tools` is
already there. Safe to re-run.

In [ ]:
import importlib
import os
import shutil
import site
import subprocess
import sys

# sra-tools is bioconda-only, so we need a conda-compatible package
# manager. Workbench images vary: newer ones ship `micromamba`, older
# ones `conda`. `mamba` is sometimes a thin wrapper that can behave
# unpredictably with conda-style flags, so it sits last.
for _mgr in ("micromamba", "conda", "mamba"):
    if shutil.which(_mgr):
        PKG_MANAGER = _mgr
        break
else:
    raise RuntimeError(
        "No conda-compatible package manager (micromamba, conda, mamba) "
        "found in PATH. sra-tools must come from bioconda."
    )

if not shutil.which("prefetch"):
    print(f"Installing sra-tools via {PKG_MANAGER}...")
    # Same per-user env pattern as nb 03 / nb 06 for micromamba: install
    # into ~/nf-env with root-prefix ~/.micromamba. Some Workbench images
    # lock down /opt/micromamba/pkgs/cache for the jupyter user, which
    # breaks a system-wide `-n base` install. A per-user env writes
    # everything under $HOME.
    if PKG_MANAGER == "micromamba":
        NF_ENV = os.path.expanduser("~/nf-env")
        MAMBA_ROOT = os.path.expanduser("~/.micromamba")
        verb = "install" if os.path.exists(os.path.join(NF_ENV, "conda-meta")) else "create"
        _install_cmd = [
            "micromamba", verb, "-y",
            "--prefix", NF_ENV,
            "--root-prefix", MAMBA_ROOT,
        ]
    else:
        _install_cmd = [PKG_MANAGER, "install", "-y"]
    _install_cmd += ["-c", "bioconda", "sra-tools"]
    subprocess.run(_install_cmd, check=True)

    # For the per-user env, add its bin dir to PATH so the next cells
    # find `prefetch` and `fasterq-dump`.
    if PKG_MANAGER == "micromamba":
        env_bin = os.path.join(NF_ENV, "bin")
        if env_bin not in os.environ["PATH"].split(os.pathsep):
            os.environ["PATH"] = env_bin + os.pathsep + os.environ["PATH"]
    print("sra-tools installed.")
else:
    print("sra-tools already installed.")

# Biopython is pure Python; pip is simpler and works regardless of which
# conda-compatible manager is on the image. --user writes to a per-user
# site-packages dir so this works even if the base env's site-packages
# is read-only (common on micromamba images).
try:
    importlib.import_module("Bio")
    print("Biopython is already installed.")
except ImportError:
    print("Installing Biopython via pip...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--user",
         "biopython"],
        check=True,
    )
    # Some kernels don't auto-include the user-site dir in sys.path if
    # it didn't exist at kernel startup (pip --user creates it on first
    # use). Add it explicitly so the `from Bio import Entrez` in the
    # GenBank cell below succeeds.
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.insert(0, user_site)
    importlib.invalidate_caches()
    print("Biopython installed.")

## What is SRA?

The Sequence Read Archive is NCBI's public repository of raw
sequencing reads. Every public sequencing study deposits raw FASTQs
(or vendor-specific formats convertible to FASTQ) into SRA.

Accession formats:

- `SRR<digits>`: one sequencing run (e.g. `SRR2584863`)
- `SRP<digits>`: a project containing multiple runs
- `SRX<digits>`: an experiment definition
- `SRS<digits>`: a sample

The European mirror uses `ERR`, `ERP`, and so on. NCBI cross-
references all of them.

For paired-end runs, `fasterq-dump` produces two files:
`<acc>_1.fastq` and `<acc>_2.fastq`. For single-end runs, just
`<acc>.fastq`.

In [ ]:
import subprocess

print(f"Prefetching {ACCESSION}...")
try:
    subprocess.run(
        ["prefetch", "--output-directory", OUTPUT_DIR, ACCESSION],
        check=True,
    )
    print("\nPrefetch complete.")
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"prefetch failed for {ACCESSION!r} (exit {e.returncode}). "
        "Common causes: invalid accession ID, no network from this VM, "
        "or NCBI temporarily rate-limiting. Try the ACCESSION in the SRA "
        "web UI to confirm it exists: "
        f"https://www.ncbi.nlm.nih.gov/sra/?term={ACCESSION}"
    ) from e

In [ ]:
import subprocess

print(f"Converting {ACCESSION} to FASTQ...")
try:
    subprocess.run(
        ["fasterq-dump",
         "--outdir", OUTPUT_DIR,
         "--threads", "4",
         ACCESSION],
        check=True,
    )
    print("\nConversion complete.")
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"fasterq-dump failed for {ACCESSION!r} (exit {e.returncode}). "
        "Most often this means the prefetch step (above) didn't actually "
        f"land the .sra file in {OUTPUT_DIR}. Check the file listing in "
        "the next cell, or re-run the prefetch cell."
    ) from e

In [ ]:
import os

print(f"Files now in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    full = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(full):
        size_mb = os.path.getsize(full) / 1024 / 1024
        print(f"  {size_mb:>7.1f} MB  {f}")
    else:
        print(f"  [DIR]      {f}")

# Peek at one fastq file
fastq_path = os.path.join(OUTPUT_DIR, f"{ACCESSION}_1.fastq")
if not os.path.exists(fastq_path):
    fastq_path = os.path.join(OUTPUT_DIR, f"{ACCESSION}.fastq")
if os.path.exists(fastq_path):
    print(f"\nFirst 8 lines of {fastq_path}:")
    with open(fastq_path) as f:
        for i, line in enumerate(f):
            if i >= 8: break
            print(f"  {line.rstrip()}")
else:
    print(
        f"\n⚠ Couldn't find {ACCESSION}_1.fastq or {ACCESSION}.fastq in "
        f"{OUTPUT_DIR}. fasterq-dump may have written under a different "
        "name — check the file listing above."
    )

## ENA: simpler downloads via plain HTTPS

The European Nucleotide Archive holds the same data as SRA, but
exposes every run as a direct HTTPS URL. No toolkit needed. Often
faster and simpler than `prefetch`/`fasterq-dump`, especially in
scripts.

The URL follows this pattern:

```
https://ftp.sra.ebi.ac.uk/vol1/fastq/<first6>/[<subdir>/]<acc>/<acc>_<1or2>.fastq.gz
```

`<first6>` is the first six characters of the accession. `<subdir>` is
a middle directory ENA uses for accessions 10 chars or longer: the
trailing digits past position 9, zero-padded to 3. So `SRR123456` has
no `<subdir>`, but `SRR2584863` (10 chars) uses `003`.

In [ ]:
import requests
import os

ena_dir = os.path.join(OUTPUT_DIR, "from-ena")
os.makedirs(ena_dir, exist_ok=True)

first6 = ACCESSION[:6]
# ENA inserts a middle subdirectory for accessions 10+ chars long. For
# SRR2584863 (10 chars) the subdir is '003' (trailing digit zero-padded
# to 3). Longer accessions use more trailing digits, same zero-pad rule.
if len(ACCESSION) <= 9:
    middle = ""
else:
    middle = ACCESSION[-(len(ACCESSION) - 9):].zfill(3) + "/"

# Try paired-end (_1.fastq.gz) first, fall back to single-end if 404.
candidate_urls = [
    f"https://ftp.sra.ebi.ac.uk/vol1/fastq/{first6}/{middle}{ACCESSION}/{ACCESSION}_1.fastq.gz",
    f"https://ftp.sra.ebi.ac.uk/vol1/fastq/{first6}/{middle}{ACCESSION}/{ACCESSION}.fastq.gz",
]

resp = None
url_used = None
for url in candidate_urls:
    print(f"Trying {url}...")
    r = requests.get(url, stream=True, timeout=300)
    if r.status_code == 200:
        resp = r
        url_used = url
        break
    print(f"  {r.status_code} — trying next pattern")

if resp is None:
    raise RuntimeError(
        f"ENA download failed for {ACCESSION!r}. None of these URLs returned 200:\n"
        + "\n".join(f"  {u}" for u in candidate_urls)
        + f"\n\nFor older accessions or unusual runs, browse "
          f"https://www.ebi.ac.uk/ena/browser/view/{ACCESSION} for the actual FTP paths."
    )

out_filename = url_used.rsplit("/", 1)[-1]
out_path = os.path.join(ena_dir, out_filename)
with open(out_path, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        f.write(chunk)
size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f"\nDownloaded {size_mb:.1f} MB to {out_path}")

## Sanity check: do the ENA and SRA files agree?

Both archives store the same reads, but they use different FASTQ
header conventions. SRA's `fasterq-dump` writes Casava 1.8 style
(`@id length=150` plus the identifier repeated on the `+` line).
ENA uses Casava 1.4 style (`@id/1` for the forward read, bare `+`
separator). The sequence and quality scores are byte-identical when
both come from the same underlying run; only the identifier and
separator format differ.

The cell below compares the actual data (lines 2 and 4) and notes
the expected header-format mismatch separately.

In [ ]:
import gzip
import os

# The ENA file is whatever .fastq.gz we just downloaded into ena_dir.
ena_files = sorted(f for f in os.listdir(ena_dir) if f.endswith(".fastq.gz"))
if not ena_files:
    raise RuntimeError(f"No .fastq.gz file found in {ena_dir}. Run the ENA cell first.")

ena_first_record = []
with gzip.open(os.path.join(ena_dir, ena_files[0]), "rt") as f:
    for i, line in enumerate(f):
        if i >= 4: break
        ena_first_record.append(line.rstrip())

sra_first_record = []
sra_path = os.path.join(OUTPUT_DIR, f"{ACCESSION}_1.fastq")
if not os.path.exists(sra_path):
    sra_path = os.path.join(OUTPUT_DIR, f"{ACCESSION}.fastq")
if os.path.exists(sra_path):
    with open(sra_path) as f:
        for i, line in enumerate(f):
            if i >= 4: break
            sra_first_record.append(line.rstrip())
else:
    print(f"⚠ No SRA FASTQ found in {OUTPUT_DIR} — run the SRA cells above first.")
    print("  (Skipping comparison; just showing the ENA record.)\n")

print("First record from ENA:")
for line in ena_first_record:
    print(f"  {line}")
if sra_first_record:
    print("\nFirst record from SRA:")
    for line in sra_first_record:
        print(f"  {line}")
    # Compare the actual data (sequence on line 2, quality on line 4). Lines 1
    # and 3 are identifier/separator — SRA and ENA use different conventions
    # there (length=150 vs /1, repeated id vs bare +), but the data lines are
    # byte-identical when both come from the same underlying run.
    sequence_match = ena_first_record[1] == sra_first_record[1]
    quality_match  = ena_first_record[3] == sra_first_record[3]
    headers_match  = ena_first_record[0] == sra_first_record[0]
    print(f"\nSequence match:  {sequence_match}")
    print(f"Quality match:   {quality_match}")
    print(f"Headers match:   {headers_match}  (expected False — Casava 1.4 vs 1.8)")
    if sequence_match and quality_match:
        print("\n✓ ENA and SRA agree on the underlying data.")
    else:
        print("\n⚠ Data mismatch — unexpected. Check that both files are the same accession.")

## Reference genomes from GenBank

Most pathogen-genomic work needs a reference genome to align reads
against. GenBank has annotated reference sequences for nearly every
named organism, and you can pull them via NCBI's Entrez API.
Biopython wraps that API cleanly.

Below we fetch a single Influenza A segment as an example. Real
analyses usually pull a specific strain reference (for SARS-CoV-2 it
might be `MN908947.3`), or build a custom reference by stitching
multiple records together.

In [ ]:
from Bio import Entrez
import os

# Entrez requires you to identify yourself.
# Use the gcloud account from earlier if available, else a safe fallback.
try:
    Entrez.email = active_account
except NameError:
    Entrez.email = "apgap-user@asu.edu"
    print("⚠ active_account not set; using fallback Entrez email.")

# Example: Influenza A segment 4 (hemagglutinin) reference
handle = Entrez.efetch(
    db="nuccore",
    id="CY121687.1",  # Influenza A virus segment 4 (H1N1)
    rettype="fasta",
    retmode="text",
)
ref_fasta = handle.read()
handle.close()

ref_path = os.path.join(OUTPUT_DIR, "reference.fasta")
with open(ref_path, "w") as f:
    f.write(ref_fasta)

print(ref_fasta[:500])
print(f"\nFull reference written to {ref_path}")

## Pushing downloaded data into your project bucket

Once data is on the VM, you'll usually want to push it into your
project's analytical-dataset bucket so a pipeline run can pick it
up (or just so the APGAP web app sees it).

The cell is commented out. Uncomment it with a real destination URI
when you're ready.

In [ ]:
# import subprocess
# DEST_URI = "gs://h1n1-analytical-dataset-21-1779319608-0ef929/imported/"  # change me
# subprocess.run(
#     ["gsutil", "-m", "cp", "-r", f"{OUTPUT_DIR}/", DEST_URI],
#     check=True,
# )
# print(f"Uploaded contents of {OUTPUT_DIR} to {DEST_URI}")

## Cleaning up local files

Workbench instances ship with around 100 GB of local disk, but
downloads pile up fast. Once you've uploaded what you want to keep,
delete the local copies.

In [ ]:
# import shutil
# shutil.rmtree(OUTPUT_DIR)
# print(f"Deleted {OUTPUT_DIR}")

## Where to go next

- Want to analyze what you just downloaded? Copy it to your project
  bucket (cell above), then open
  [02-read-your-data.ipynb](02-read-your-data.ipynb).
- Want to run a pipeline on it? Open
  [03-launch-a-pipeline.ipynb](03-launch-a-pipeline.ipynb).
- Back to [getting-started](01-getting-started.ipynb).

Background reading:

- [sra-tools on GitHub](https://github.com/ncbi/sra-tools)
- [ENA browser](https://www.ebi.ac.uk/ena/browser/home)
- [Biopython Entrez tutorial](https://biopython.org/DIST/docs/tutorial/Tutorial.html#chapter:entrez)